# AtendeAI — Fase 3: memória de curto e longo prazo (o agente vira chatbot)
Continua as Fases 1 e 2. **Uso de IA:** ver `docs/uso_de_ia.md`.

**Evidências desta fase (checklist do enunciado):**
- [ ] Curto vs. longo prazo no domínio, pela taxonomia CoALA (seção 1)
- [ ] Estratégia de janela de contexto: sumarização + busca vetorial (seção 2)
- [ ] Comparação de ao menos duas ferramentas de memória e decisão (seção 3)
- [ ] Chatbot com `ConversationBufferMemory` (seção 4)
- [ ] Reconhecimento de intenção e extração de entidades em interação real (seção 6)
- [ ] Conversa multi-turno com contexto mantido e memória de longo prazo entre sessões (seções 5 e 7)

> **Custo:** a sessão completa deste notebook faz cerca de 25 chamadas ao LLM. Rode **uma vez** com o LLM real para gerar a evidência; para desenvolver, use `LLM_MODE=dummy`.

## 1. Vocabulário de memória do AtendeAI (taxonomia CoALA)
CoALA (*Cognitive Architectures for Language Agents*) distingue a memória de **trabalho** (o que está em jogo agora) das memórias de **longo prazo**: semântica, episódica e procedural.

| Tipo | O que é no AtendeAI | Exemplo | Onde vive | Como é lida / atualizada |
|---|---|---|---|---|
| **Curto prazo (trabalho)** | Mensagens da sessão e o estado da conversa (slots: pedido, CEP, produto; última intenção) | "Recebi ele há 3 dias" só faz sentido com a mensagem anterior sobre o tênis | `ConversationBufferMemory` + `slots` | Lida inteira a cada turno; resumida quando cresce (seção 2) |
| **Longo prazo, semântica** | Fatos estáveis sobre o cliente | `CEP padrão do cliente: 80010-000` | SQLite (`memorias`, tipo `semantica`) | **Sempre** carregada (perfil); atualizada por *upsert* via `chave` |
| **Longo prazo, episódica** | Eventos de atendimentos passados | `2026-11-08: cliente consultou troca ou devolução (produto: tênis)` | SQLite, tipo `episodica` | Recuperada por **similaridade vetorial** (top-k) com a mensagem atual; só acrescenta |
| **Longo prazo, procedural** | Como a Lia deve atender | O prompt PTCF e as regras de atendimento | `src/prompts_react.py` (versionado no Git) | **Não é reescrita sozinha**: mudanças de prompt são decisões humanas validadas por medição (ver o experimento A/B da Fase 2) |

## 2. Estratégia de janela de contexto
1. **Buffer completo** até `max_mensagens` (12 em produção, 8 neste notebook para a demonstração). Atendimentos são conversas curtas, então o buffer completo é a opção mais simples e sem perda.
2. **Sumarização ao estourar:** as mensagens antigas viram um resumo (1 chamada de LLM) e as últimas `manter=4` ficam literais, formando uma **janela deslizante sobre o resumo** (`src/contexto.py`).
3. **Fatos que não podem se perder** sobrevivem ao resumo por dois caminhos independentes: os **slots** (reinjetados no contexto de todo turno) e a **memória de longo prazo**.
4. **Recuperação e atualização vetorial:** a cada turno, as memórias do cliente são buscadas por similaridade de cosseno entre embeddings (perfil sempre; episódicas por top-k com limiar). Um fato novo com a mesma `chave` **substitui** o anterior (upsert), e um evento idêntico no mesmo dia não duplica.
5. **Limites conhecidos:** não há esquecimento automático (TTL) nem tratamento de dados pessoais (LGPD) além do isolamento por cliente (ver seção 8).

## 3. Ferramentas especializadas de memória: comparação e decisão
*Resumo a partir de fontes secundárias (blogs técnicos e páginas de comparação, algumas publicadas por fornecedores; ver lista ao final). Confirme na documentação oficial e cite-a no relatório.*

| | **LangMem** | **Mem0** | **Letta (MemGPT)** |
|---|---|---|---|
| **O que é** | Biblioteca de *primitivas* de memória sobre o LangGraph | Camada de memória plug-and-play (`pip install`) | **Runtime de agentes**, não uma biblioteca |
| **Arquitetura** | Memória semântica (coleções e perfis), episódica (exemplos) e **procedural** (otimização das instruções do agente), com gerenciador em segundo plano | Extrai fatos das conversas com um LLM, guarda por usuário/sessão e recupera de forma híbrida (semântica, palavra-chave e entidades) | Memória em camadas estilo sistema operacional (contexto como "RAM", armazenamento externo como "disco") que **o próprio agente edita** via chamadas de ferramenta |
| **Melhor caso de uso** | Montar a própria camada de memória dentro do ecossistema LangGraph | Assistentes em produção com memória por usuário e pouco atrito, independentes de framework | Agentes autônomos e de longa duração que curam a própria memória |
| **Pontos de atenção** | Acoplamento ao LangGraph, API em evolução, documentação descrita como esparsa | Cada extração custa chamadas de LLM (custo e latência); benchmarks citados vêm do próprio fornecedor | Exige adotar o runtime e um servidor; memória guiada pelo modelo é menos previsível |

**Decisão: nenhuma entra no AtendeAI agora.**
- O agente usa `AgentExecutor` (`initialize_agent`), que o enunciado exige. O LangMem pede LangGraph e o Letta pede trocar de runtime.
- O Mem0 adicionaria chamadas de LLM de extração a cada turno, e a cota diária do modelo gratuito (500 pedidos) e o prazo (13/11) tornam isso caro.
- O domínio tem poucos fatos a lembrar (CEP padrão, histórico de consultas), e **SQLite + similaridade vetorial + regras de extração** cobrem isso com custo zero de LLM.
- A camada própria segue a mesma divisão conceitual (semântica, episódica, procedural), então pode ser **trocada pelo Mem0 ou pelo LangMem** na próxima disciplina, quando houver outros agentes ao redor.

**Trade-off aceito:** a extração de fatos por regras é frágil (só reconhece "meu CEP é ..."); um extrator com LLM, como o do Mem0, generalizaria melhor, ao custo citado acima.

*Fontes consultadas:* mem0.ai (blog de benchmark, do fornecedor); rickhigh.substack.com (comparação, 15/07/2026); memoryplugin.com (panorama de ferramentas); evermind.ai (alternativas ao LangMem); aitoolsatlas.ai (LangMem vs Letta).

In [ ]:
import os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import pandas as pd
from src.llm_factory import get_llm
from src.callbacks import LogHandler
from src.memoria_longa import MemoriaLongoPrazo
from src.chatbot import AtendeAIChatbot

llm = get_llm()
print("Modo:", os.getenv("LLM_MODE", "dummy"), "|", type(llm).__name__)

CLIENTE = 1
mem = MemoriaLongoPrazo(ROOT / "data" / "memoria_demo.db")   # banco só da demonstração
mem.limpar(CLIENTE)                                           # começa do zero, para ser reproduzível
handler = LogHandler()

## 4. O chatbot: componentes
Cada turno: (1) reconhece intenção e entidades; (2) atualiza os slots e trata continuações; (3) recupera memórias de longo prazo; (4) roda o agente ReAct com `ConversationBufferMemory`; (5) grava memórias novas e gerencia a janela de contexto. Detalhes em `src/chatbot.py`.

In [ ]:
bot = AtendeAIChatbot(llm, CLIENTE, memoria_longa=mem, handler=handler, max_mensagens=8, manter=4, verbose=False)
print("Cliente:", bot.cliente)
print("Memória de curto prazo:", type(bot.memoria).__name__)
print("Janela de contexto: máx.", bot.contexto.max_mensagens, "mensagens, mantendo", bot.contexto.manter, "literais após resumir")

## 5. Conversa multi-turno (sessão 1)
Cenário que exige **decompor** o pedido e **lembrar** do que foi dito: informar o CEP, perguntar sobre frete, pedir devolução, informar quando recebeu e voltar ao prazo usando "no meu CEP".

In [ ]:
MENSAGENS = [
    "Oi! Meu CEP é 80010-000. Qual o prazo e o frete?",
    "E esse frete é grátis?",
    "Quero devolver o tênis que comprei, dá?",
    "Recebi ele há 3 dias.",
    "E qual o prazo de entrega de um novo no meu CEP?",
]
for m in MENSAGENS:
    t = bot.responder(m)
    print(f"Cliente: {m}")
    print(f"Lia:     {t.resposta}")
    print(f"   [intenção={t.intencao_efetiva}{' (continuação)' if t.continuacao else ''} | ferramentas={t.ferramentas or 'nenhuma'} "
          f"| erros de formato={t.erros_formato} | {t.tempo_s}s]" + (f" | ERRO: {t.erro}" if t.erro else ""))
    print()

## 6. Reconhecimento de intenção e extração de entidades (a partir da conversa acima)

In [ ]:
pd.DataFrame([{
    "mensagem": t.mensagem[:52],
    "intenção (modelo)": t.intencao,
    "intenção efetiva": t.intencao_efetiva,
    "continuação": t.continuacao,
    "entidades extraídas": {k: v for k, v in t.entidades.items() if v},
    "estado da sessão (slots)": {k: v for k, v in t.slots.items() if v},
} for t in bot.turnos])

**Como ler:** em "Recebi ele há 3 dias." o modelo não vê intenção própria (`outro`), mas a mensagem é curta e a conversa vinha de uma devolução, então a intenção efetiva **herda o assunto** (`continuação=True`). Em "...no meu CEP?" a entidade `cep` não aparece na mensagem, mas está nos **slots**, vindo de um turno anterior.

### 6.1 Memória de curto prazo e gerenciamento da janela

In [ ]:
print("Resumo gerado em algum turno?", [i + 1 for i, t in enumerate(bot.turnos) if t.resumo_gerado])
print()
print(bot.memoria.load_memory_variables({})["chat_history"][:1800])

## 7. Memória de longo prazo: o que foi guardado e uma nova sessão

In [ ]:
pd.DataFrame(mem.listar(CLIENTE))[["tipo", "chave", "texto", "criado_em"]]

In [ ]:
# NOVA SESSÃO: objeto novo (curto prazo vazio, sem slots), mesmo cliente e mesmo banco de memória
bot2 = AtendeAIChatbot(llm, CLIENTE, memoria_longa=mem, handler=handler, max_mensagens=8, manter=4, verbose=False)
print("slots no início da sessão 2:", bot2.slots)

t6 = bot2.responder("Qual o prazo de entrega para o meu CEP de sempre?")
print("Cliente:", t6.mensagem)
print("Lia:    ", t6.resposta)
print("memórias de longo prazo usadas:", t6.memorias)
print("ferramentas:", t6.ferramentas, "| erro:", t6.erro)

## 8. Resumo, limitações e próximos passos

| Componente | Escolha | Justificativa | Limitação |
|---|---|---|---|
| Curto prazo | `ConversationBufferMemory` + slots + continuação de intenção | Exigência do enunciado; conversas de atendimento são curtas; slots garantem que pedido, CEP e produto não se percam | Slots só guardam 3 entidades |
| Janela de contexto | Buffer até o limite, depois sumarização + últimas 4 mensagens | Sem perda até o limite e custo de 1 chamada de LLM só quando estoura | O resumo pode perder detalhes (por isso os slots e a memória longa) |
| Longo prazo | SQLite + cosseno entre embeddings; perfil sempre, episódico por top-k | Custo zero de LLM, 100% auditável e trocável por Mem0 ou LangMem | Extração por regras ("meu CEP é ..."); sem esquecimento automático; embeddings lexicais no modo `hashing` |
| Contexto no prompt | Slots e memórias entram como um trecho entre parênteses na pergunta | Não altera o prompt PTCF validado na Fase 2 (lição do experimento A/B) | Aumenta o texto de cada pergunta |
| Privacidade | Memória isolada por `cliente_id` | Um cliente nunca recupera memória de outro | Dados pessoais (LGPD): faltam consentimento, retenção e exclusão sob demanda além do `limpar()` |

- **Para o relatório:** guarde as saídas das seções 5, 6, 6.1 e 7 (a "sessão 2" é a evidência de memória de longo prazo).
- **Para o vídeo:** use `python -m src.cli --cliente 1 --trace` e repita este roteiro ao vivo.
- **Fase 4:** SQL (text-to-query) e o agente de recuperação com diagnóstico de falhas.